# NB04 — Presentation Demo

**Runs on Kaggle T4 GPU.**

**Datasets to attach:**
- `dummyirl/sam3-weights`
- `dummyirl/darmstadt-dop20`

**Live demo flow (for Praktikum presentation):**
1. **Section A** — Display a grid of Hessen tiles. Audience picks 2 by index.
2. **Section B** — Run A: standard Potsdam vocabulary on the chosen tiles (shows domain gap)
3. **Section C** — Audience describes what they see → edit `USER_CLASSES` live
4. **Section D** — Run B: audience's vocabulary → 3-panel comparison

**The point:** same frozen SAM3, same weights, same image — different text prompts → different segmentation.

## 1 — Environment setup

In [ ]:
import os

!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda_installer.sh
!bash /tmp/miniconda_installer.sh -b -p /kaggle/working/miniconda

os.environ.pop("PYTHONPATH", None)
os.environ["PATH"] = "/kaggle/working/miniconda/bin:" + os.environ["PATH"]

!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!conda --version

In [ ]:
!/kaggle/working/miniconda/bin/conda create -n segearth python=3.10 -y

In [ ]:
!conda run -n segearth pip install torch==2.4.0 torchvision==0.19.0 -q

In [ ]:
!conda run -n segearth pip install openmim -q
!conda run -n segearth mim install "mmcv==2.2.0" -q
!conda run -n segearth pip install "mmsegmentation==1.2.2" -q

In [ ]:
%%bash
source /kaggle/working/miniconda/bin/activate segearth
python - << 'EOF'
import pathlib
f = pathlib.Path("/kaggle/working/miniconda/envs/segearth/lib/python3.10/site-packages/mmseg/__init__.py")
f.write_text(f.read_text().replace("MMCV_MAX = '2.2.0'", "MMCV_MAX = '2.3.0'"))
print("Patched MMCV_MAX → 2.3.0")
EOF
pip install numpy==1.26.4 -q

In [ ]:
%%bash
source /kaggle/working/miniconda/bin/activate segearth
python - << 'EOF'
import mmcv; print("MMCV:", mmcv.__version__)
from mmseg.structures import SegDataSample; print("MMSEG OK")
import torch; print("CUDA:", torch.cuda.is_available())
EOF

## 2 — Clone our fork

In [ ]:
import subprocess, os
from pathlib import Path

REPO = Path("/kaggle/working/SegEarth-OV-3")

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(
        ["git", "clone", "--depth=1",
         "https://github.com/HarishDeepak/SegEarth-OV-3.git", str(REPO)],
        check=True)

os.chdir(REPO)
!conda run -n segearth pip install -r requirements.txt -q
print("Ready.")

## Section A — Display tile grid

Show this to the audience. Ask them to pick 2 tiles by index number.

In [ ]:
%%bash
export MPLBACKEND=Agg
source /kaggle/working/miniconda/bin/activate segearth
cd /kaggle/working/SegEarth-OV-3

python - << 'EOF'
import numpy as np, os
os.environ["MPLBACKEND"] = "Agg"
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

INPUT_FOLDER = "/kaggle/input/darmstadt-dop20/"
TILE_EXTS = [".tif", ".tiff", ".jpg", ".jpeg", ".png"]
base = Path(INPUT_FOLDER)
all_tiles = sorted(p for ext in TILE_EXTS for p in base.glob(f"*{ext}"))
if not all_tiles:
    all_tiles = sorted(p for ext in TILE_EXTS for p in base.glob(f"**/*{ext}"))

print(f"Available tiles (0-{len(all_tiles)-1}):")
for i, p in enumerate(all_tiles):
    print(f"  [{i}] {p.name}")

n = min(len(all_tiles), 12)
cols = 4
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
axes = np.array(axes).flatten()
for i in range(n):
    img = Image.open(all_tiles[i]).convert("RGB")
    axes[i].imshow(img)
    axes[i].set_title(f"[{i}] {all_tiles[i].stem[:20]}", fontsize=8)
    axes[i].axis("off")
for j in range(n, len(axes)):
    axes[j].axis("off")
plt.tight_layout()
os.makedirs("output", exist_ok=True)
plt.savefig("output/tile_grid.png", dpi=100, bbox_inches="tight")
plt.close()
print("\nSaved: output/tile_grid.png")
print("→ Show this to the audience and ask them to pick 2 tile indices.")
EOF

## Section B — Run A: Potsdam vocabulary

**Edit `TILE_IDX` below** to the 2 tiles the audience picked. Then run this cell.

In [ ]:
TILE_IDX = [0, 1]  # ← CHANGE to audience's chosen indices

In [ ]:
%%bash
export MPLBACKEND=Agg DEMO_TILE_IDX="{','.join(map(str, TILE_IDX))}"
source /kaggle/working/miniconda/bin/activate segearth
cd /kaggle/working/SegEarth-OV-3

python - << 'EOF'
import torch, numpy as np, os
os.environ["MPLBACKEND"] = "Agg"
torch.manual_seed(42)

from pathlib import Path
from PIL import Image
from torchvision import transforms
from mmseg.structures import SegDataSample
from segearthov3_segmentor import SegEarthOV3Segmentation

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

TILE_IDX = list(map(int, os.environ.get("DEMO_TILE_IDX", "0,1").split(",")))
INPUT_FOLDER = "/kaggle/input/darmstadt-dop20/"
TILE_EXTS = [".tif", ".tiff", ".jpg", ".jpeg", ".png"]
base = Path(INPUT_FOLDER)
all_tiles = sorted(p for ext in TILE_EXTS for p in base.glob(f"*{ext}"))
if not all_tiles:
    all_tiles = sorted(p for ext in TILE_EXTS for p in base.glob(f"**/*{ext}"))
chosen = [all_tiles[i] for i in TILE_IDX if i < len(all_tiles)]

POTSDAM_NAMES  = ["impervious surface", "building", "low vegetation", "tree", "car", "clutter"]
POTSDAM_COLORS = np.array([[128,0,0],[0,0,255],[0,255,0],[0,128,0],[255,255,0],[255,0,0]], dtype=np.uint8)

model = SegEarthOV3Segmentation(
    type="SegEarthOV3Segmentation",
    classname_path="./configs/cls_potsdam.txt",
    prob_thd=0.1, bg_idx=5, confidence_threshold=0.1,
    slide_stride=128, slide_crop=256,
)

os.makedirs("output", exist_ok=True)
for img_path in chosen:
    img = Image.open(img_path).convert("RGB")
    ds = SegDataSample()
    ds.set_metainfo({"img_path": str(img_path), "ori_shape": img.size[::-1]})
    result = model.predict(transforms.ToTensor()(img).unsqueeze(0).to("cuda"), [ds])
    seg = result[0].pred_sem_seg.data.cpu().numpy().squeeze(0)
    seg_rgb = POTSDAM_COLORS[np.clip(seg, 0, 5)]
    fig, ax = plt.subplots(1, 2, figsize=(14, 7))
    ax[0].imshow(img); ax[0].axis("off"); ax[0].set_title("RGB")
    ax[1].imshow(img); ax[1].imshow(seg_rgb, alpha=0.5)
    ax[1].axis("off"); ax[1].set_title("Run A — Potsdam vocab (shows domain gap)")
    fig.legend(handles=[Patch(facecolor=c/255., label=n) for n, c in zip(POTSDAM_NAMES, POTSDAM_COLORS)],
               loc="lower center", ncol=6, bbox_to_anchor=(0.5, -0.02))
    plt.tight_layout(rect=[0, 0.05, 1, 1])
    out = f"output/{img_path.stem}_runA.png"
    plt.savefig(out, dpi=150, bbox_inches="tight"); plt.close()
    print(f"Saved: {out}")
EOF

## Section C — Audience's vocabulary

Show the audience the RGB tiles from Section A.  
Ask: *"What do you see in this image?"*  
Edit `USER_CLASSES` below. Each key = class name. Each value = list of synonyms.

Then run Section D.

In [ ]:
# ← Edit this live during the presentation
USER_CLASSES = {
    "agricultural field": ["agricultural field", "farmland", "crop field", "ploughed field"],
    "forest":             ["forest", "woodland", "dense trees", "tree canopy"],
    "building":           ["building", "rooftop", "house", "village structure"],
    "road":               ["road", "path", "dirt track", "country road"],
    "water":              ["water", "river", "stream", "pond"],
    "background":         ["background", "open land", "bare soil"],
}

# Write cls_user.txt from the dict
from pathlib import Path
cls_path = Path("/kaggle/working/SegEarth-OV-3/configs/cls_user.txt")
with open(cls_path, "w") as f:
    for synonyms in USER_CLASSES.values():
        f.write(", ".join(synonyms) + "\n")
print(f"Written {cls_path}:")
print(cls_path.read_text())

## Section D — 3-panel comparison: RGB | Run A (Potsdam) | Run B (Audience)

In [ ]:
%%bash
export MPLBACKEND=Agg
source /kaggle/working/miniconda/bin/activate segearth
cd /kaggle/working/SegEarth-OV-3

python - << 'EOF'
import torch, numpy as np, os
os.environ["MPLBACKEND"] = "Agg"
torch.manual_seed(42)

from pathlib import Path
from PIL import Image
from torchvision import transforms
from mmseg.structures import SegDataSample
from segearthov3_segmentor import SegEarthOV3Segmentation
import matplotlib.cm as cm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

TILE_IDX = list(map(int, os.environ.get("DEMO_TILE_IDX", "0,1").split(",")))
INPUT_FOLDER = "/kaggle/input/darmstadt-dop20/"
TILE_EXTS = [".tif", ".tiff", ".jpg", ".jpeg", ".png"]
base = Path(INPUT_FOLDER)
all_tiles = sorted(p for ext in TILE_EXTS for p in base.glob(f"*{ext}"))
if not all_tiles:
    all_tiles = sorted(p for ext in TILE_EXTS for p in base.glob(f"**/*{ext}"))
chosen = [all_tiles[i] for i in TILE_IDX if i < len(all_tiles)]

cls_path = "./configs/cls_user.txt"
user_names = [line.split(",")[0].strip() for line in open(cls_path) if line.strip()]
n_cls = len(user_names)
cmap = cm.get_cmap("tab10", n_cls)
COLOR_B = (np.array([cmap(i)[:3] for i in range(n_cls)]) * 255).astype(np.uint8)

POTSDAM_NAMES  = ["impervious surface", "building", "low vegetation", "tree", "car", "clutter"]
POTSDAM_COLORS = np.array([[128,0,0],[0,0,255],[0,255,0],[0,128,0],[255,255,0],[255,0,0]], dtype=np.uint8)

model_A = SegEarthOV3Segmentation(
    type="SegEarthOV3Segmentation", classname_path="./configs/cls_potsdam.txt",
    prob_thd=0.1, bg_idx=5, confidence_threshold=0.1, slide_stride=128, slide_crop=256)
model_B = SegEarthOV3Segmentation(
    type="SegEarthOV3Segmentation", classname_path=cls_path,
    prob_thd=0.1, bg_idx=n_cls-1, confidence_threshold=0.1, slide_stride=128, slide_crop=256)

os.makedirs("output", exist_ok=True)
for img_path in chosen:
    img = Image.open(img_path).convert("RGB")
    img_t = transforms.ToTensor()(img).unsqueeze(0).to("cuda")

    def run(model):
        ds = SegDataSample()
        ds.set_metainfo({"img_path": str(img_path), "ori_shape": img.size[::-1]})
        return model.predict(img_t, [ds])[0].pred_sem_seg.data.cpu().numpy().squeeze(0)

    rgb_A = POTSDAM_COLORS[np.clip(run(model_A), 0, 5)]
    rgb_B = COLOR_B[np.clip(run(model_B), 0, n_cls-1)]

    fig, axes = plt.subplots(1, 3, figsize=(21, 7))
    axes[0].imshow(img);       axes[0].axis("off"); axes[0].set_title("RGB", fontsize=13)
    axes[1].imshow(img); axes[1].imshow(rgb_A, alpha=0.5)
    axes[1].axis("off"); axes[1].set_title("Run A — Potsdam vocab", fontsize=13)
    axes[2].imshow(img); axes[2].imshow(rgb_B, alpha=0.5)
    axes[2].axis("off"); axes[2].set_title("Run B — Your vocabulary", fontsize=13)
    fig.legend(handles=[Patch(facecolor=c/255., label=n) for n, c in zip(user_names, COLOR_B)],
               loc="lower center", ncol=n_cls, bbox_to_anchor=(0.5, -0.02), fontsize=10)
    plt.suptitle(f"{img_path.stem} — same frozen model, same weights, different text prompts", fontsize=11)
    plt.tight_layout(rect=[0, 0.06, 1, 0.97])
    out = f"output/{img_path.stem}_comparison.png"
    plt.savefig(out, dpi=150, bbox_inches="tight"); plt.close()
    print(f"Saved: {out}")

print("Done — open output/*_comparison.png")
EOF